In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

gdf = gpd.read_file("data_processed/final_combined/all_states_tracts.geojson")
gdf["GEOID"] = gdf["GEOID"].astype(str)

print("Loaded:", gdf.shape)
print(gdf.columns.tolist())

Loaded: (16371, 22)
['GEOID', 'population', 'households', 'median_income', 'workers', 'mean_commute_time', 'vehicles_total', 'state', 'county', 'tract', 'state_code', 'station_count', 'total_ports', 'l2_ports', 'dcfc_ports', 'dist_nearest_charger_km', 'dist_nearest_dcfc_km', 'establishments_total', 'retail_establishments', 'food_establishments', 'service_establishments', 'geometry']


In [3]:
# Project to an equal-area CRS for accurate area computation
gdf_proj = gdf.to_crs(epsg=5070)  # Albers Equal Area — good for CONUS

# Area in km²
gdf["tract_area_km2"] = gdf_proj.geometry.area / 1e6

# Population density (people per km²)
gdf["pop_density_km2"] = gdf["population"] / gdf["tract_area_km2"].replace(0, np.nan)

# Household density
gdf["hh_density_km2"] = gdf["households"] / gdf["tract_area_km2"].replace(0, np.nan)

# Log transforms (these are heavily skewed)
gdf["log_tract_area_km2"] = np.log1p(gdf["tract_area_km2"])
gdf["log_pop_density_km2"] = np.log1p(gdf["pop_density_km2"])

print(gdf[["tract_area_km2", "pop_density_km2", "hh_density_km2"]].describe())

       tract_area_km2  pop_density_km2  hh_density_km2
count    16371.000000     16371.000000    16371.000000
mean        52.524144      5022.734461     1946.626727
std        375.342000      8471.254410     3666.341989
min          0.021283         0.000000        0.000000
25%          0.736869       646.414140      226.202782
50%          1.840201      2253.929895      786.462924
75%          5.865186      4939.956074     1736.705165
max      18011.438796    110669.375220    50158.753852


In [32]:
import os
import requests
import pandas as pd
import numpy as np
import geopandas as gpd
from census import Census
from dotenv import load_dotenv
# from tqdm import tqdm

load_dotenv()

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
c = Census(CENSUS_API_KEY)

RAW_DIR = "data_raw"
PROC_DIR = "data_processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

STATE_FIPS = {
    "CA": "06",
    "CO": "08",
    "VT": "50",
    "DC": "11",
    "NY": "36"
}

In [48]:
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def safe_download(url, output_path):

    if os.path.exists(output_path):
        print("Using cached file:", output_path)
        return

    print("Downloading:", url)

    session = requests.Session()

    retries = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504]
    )

    session.mount("https://", HTTPAdapter(max_retries=retries))

    with session.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    print("Download complete.")

## ACS EXTENDED

In [34]:
def fetch_acs_extended(state_fips):

    variables = {
        "population": "B01003_001E",
        "poverty_count": "B17001_002E",
        "median_home_value": "B25077_001E",
        "median_year_built": "B25035_001E",
        "gini": "B19083_001E",
        "bachelor_plus": "B15003_022E",
        "work_from_home": "B08006_017E",
        "transit_commuters": "B08301_010E"
    }

    data = c.acs5.state_county_tract(
        list(variables.values()),
        state_fips,
        Census.ALL,
        Census.ALL
    )

    df = pd.DataFrame(data)

    df["GEOID"] = (
        df["state"] + df["county"] + df["tract"]
    ).astype(str).str.zfill(11)

    df.rename(columns={v: k for k, v in variables.items()}, inplace=True)

    df["poverty_rate"] = df["poverty_count"] / (df["population"] + 1)

    return df[["GEOID"] + list(variables.keys()) + ["poverty_rate"]]

## LODES WAC + RAC

In [49]:
def fetch_lodes(state_abbr):

    base = f"https://lehd.ces.census.gov/data/lodes/LODES8/{state_abbr.lower()}"

    wac_url = f"{base}/{state_abbr.lower()}_wac_S000_JT00_2021.csv.gz"
    rac_url = f"{base}/{state_abbr.lower()}_rac_S000_JT00_2021.csv.gz"

    wac_path = os.path.join(RAW_DIR, f"{state_abbr}_wac.csv.gz")
    rac_path = os.path.join(RAW_DIR, f"{state_abbr}_rac.csv.gz")

    safe_download(wac_url, wac_path)
    safe_download(rac_url, rac_path)

    wac = pd.read_csv(wac_path, compression="gzip")
    rac = pd.read_csv(rac_path, compression="gzip")

    # 🔥 critical zfill fix
    wac["GEOID"] = wac["w_geocode"].astype(str).str.zfill(15).str[:11]
    rac["GEOID"] = rac["h_geocode"].astype(str).str.zfill(15).str[:11]

    wac_tract = wac.groupby("GEOID")[["C000","CE01","CE02","CE03"]].sum().reset_index()
    rac_tract = rac.groupby("GEOID")[["C000","CE01","CE02","CE03"]].sum().reset_index()

    wac_tract.columns = ["GEOID","jobs_total","jobs_low","jobs_mid","jobs_high"]
    rac_tract.columns = ["GEOID","workers_total","workers_low","workers_mid","workers_high"]

    df = wac_tract.merge(rac_tract, on="GEOID", how="left")

    df["jobs_inflow_ratio"] = df["jobs_total"] / (df["workers_total"] + 1)

    return df

## SMART LOCATION

In [36]:
def fetch_smart_location():

    csv_path = os.path.join(RAW_DIR, "SmartLocationDatabaseV3.csv")

    if not os.path.exists(csv_path):
        print("Downloading Smart Location DB...")
        url = "https://edg.epa.gov/data/public/OA/EPA_SmartLocationDatabase_V3_Jan_2021_Final.csv"
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers)
        with open(csv_path, "wb") as f:
            f.write(r.content)

    df = pd.read_csv(
        csv_path,
        encoding="latin1",
        dtype={"GEOID10": str},   # 🔥 critical
        low_memory=False
    )

    df["GEOID"] = df["GEOID10"].str.zfill(11)

    df.replace(-99999, np.nan, inplace=True)

    # Keep only your states
    state_prefixes = ["06", "08", "50", "11", "36"]
    df = df[df["GEOID"].str[:2].isin(state_prefixes)]

    return df[[
        "GEOID",
        "D3B",
        "D4A",
        "D1C",
        "D2A_JPHH"
    ]]

## COMPOSITE INDEX

In [37]:
def build_composite_indexes(df):

    df["ev_adoption_propensity"] = (
        df["median_home_value"].rank(pct=True) +
        df["jobs_high"].rank(pct=True) +
        df["bachelor_plus"].rank(pct=True)
    ) / 3

    df["transit_propensity"] = (
        df["transit_commuters"].rank(pct=True) +
        df["D4A"].fillna(0).rank(pct=True)
    ) / 2

    df["home_charging_capacity_index"] = (
        df["median_home_value"].rank(pct=True) -
        df["transit_commuters"].rank(pct=True)
    )

    return df

## STATE PIPELINE

In [50]:
smart = fetch_smart_location()

all_states_df = []

for abbr, fips in STATE_FIPS.items():

    print(f"\nProcessing {abbr}")

    tracts = gpd.read_file(
        f"https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_{fips}_tract.zip"
    )

    tracts["GEOID"] = tracts["GEOID"].astype(str).str.zfill(11)

    acs = fetch_acs_extended(fips)
    lodes = fetch_lodes(abbr.lower())

    # Merge
    tracts["GEOID"] = tracts["GEOID"].astype(str).str.zfill(11)
    acs["GEOID"] = acs["GEOID"].astype(str).str.zfill(11)
    lodes["GEOID"] = lodes["GEOID"].astype(str).str.zfill(11)
    smart["GEOID"] = smart["GEOID"].astype(str).str.zfill(11)
    
    # Use D3B as road proxy
    tracts["road_density_proxy"] = tracts["D3B"]

    tracts = build_composite_indexes(tracts)

    output_path = os.path.join(PROC_DIR, f"{abbr}_enhanced.geojson")
    tracts.to_file(output_path, driver="GeoJSON")

    all_states_df.append(tracts)

    print(f"{abbr} saved.")

/var/folders/bq/hl2d1mzx4ln4s9vk722gsfv80000gn/T/ipykernel_6137/806051404.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["GEOID"] = df["GEOID10"].str.zfill(11)



Processing CA
Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/ca/ca_wac_S000_JT00_2021.csv.gz


HTTPError: 404 Client Error: Not Found for url: https://lehd.ces.census.gov/data/lodes/LODES8/ca/ca_wac_S000_JT00_2021.csv.gz

In [41]:
full_df = pd.concat(all_states_df, ignore_index=True)

print("Full dataset shape:", full_df.shape)
full_df.head()

Full dataset shape: (16386, 39)


,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,workers_high,jobs_inflow_ratio,D3B,D4A,D1C,D2A_JPHH,road_density_proxy,ev_adoption_propensity,transit_propensity,home_charging_capacity_index
0,06,029,004402,06029004402,44.02,Census Tract 44.02,G5020,S,1865739,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.324762,-0.098423
1,06,047,000802,06047000802,8.02,Census Tract 8.02,G5020,S,2321653,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.324762,-0.012597
2,06,085,501402,06085501402,5014.02,Census Tract 5014.02,G5020,S,522620,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500301,0.250192
3,06,005,000102,06005000102,1.02,Census Tract 1.02,G5020,S,456204155,7940832,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.324762,-0.013419
4,06,029,004901,06029004901,49.01,Census Tract 49.01,G5020,S,1459379,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.324762,-0.092562


In [44]:
ca = full_df[full_df["STATEFP"] == "06"]

print("Smart match rate:",
      1 - ca["D3B"].isna().mean())

Smart match rate: 0.0


In [45]:
tracts_ca = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_06_tract.zip"
)

tracts_ca["GEOID"] = tracts_ca["GEOID"].astype(str).str.zfill(11)

smart_ca = smart[smart["GEOID"].str[:2] == "06"]

len(set(tracts_ca["GEOID"]).intersection(set(smart_ca["GEOID"])))

0

In [3]:
state = STATE_FIPS["NY"]

tracts_ny = gpd.read_file(
    f"https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_{state}_tract.zip"
)

print("NY tracts shape:", tracts_ny.shape)
tracts_ny.head()

NY tracts shape: (5411, 13)


,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,+40.6923505,-073.9973434,"POLYGON ((-74.00154 40.69279, -74.00132 40.693..."
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,+40.6917206,-073.9916018,"POLYGON ((-73.99405 40.6909, -73.99374 40.6915..."
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,+40.6932903,-073.9877087,"POLYGON ((-73.99073 40.69305, -73.99045 40.693..."
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,+40.6976150,-073.9883586,"POLYGON ((-73.99141 40.69863, -73.99131 40.699..."
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,+40.6480407,-074.0159276,"POLYGON ((-74.01867 40.64741, -74.01809 40.647..."


In [4]:
def fetch_acs_extended(state_fips):
    
    variables = {
        "population": "B01003_001E",
        "poverty_count": "B17001_002E",
        "median_home_value": "B25077_001E",
        "median_year_built": "B25035_001E",
        "gini": "B19083_001E",
        "bachelor_plus": "B15003_022E",
        "work_from_home": "B08006_017E",
        "transit_commuters": "B08301_010E"
    }

    data = c.acs5.state_county_tract(
        list(variables.values()),
        state_fips,
        Census.ALL,
        Census.ALL
    )

    df = pd.DataFrame(data)
    df["GEOID"] = df["state"] + df["county"] + df["tract"]
    df.rename(columns={v: k for k, v in variables.items()}, inplace=True)

    df["poverty_rate"] = df["poverty_count"] / (df["population"] + 1)

    return df[["GEOID"] + list(variables.keys()) + ["poverty_rate"]]

In [5]:
acs_ny = fetch_acs_extended(STATE_FIPS["NY"])
print("ACS NY shape:", acs_ny.shape)
acs_ny.head()

ACS NY shape: (5396, 10)


,GEOID,population,poverty_count,median_home_value,median_year_built,gini,bachelor_plus,work_from_home,transit_commuters,poverty_rate
0,36001000100,2156.0,510.0,146900.0,1938,0.4404,194.0,0.0,185.0,0.236439
1,36001000201,2519.0,805.0,213100.0,1938,0.5514,373.0,114.0,238.0,0.319444
2,36001000202,2436.0,597.0,-666666666.0,1978,0.5344,188.0,25.0,253.0,0.244973
3,36001000301,2378.0,667.0,94400.0,1938,0.3993,97.0,0.0,387.0,0.280370
4,36001000302,3273.0,661.0,246600.0,1979,0.5830,508.0,297.0,184.0,0.201894


In [8]:
def fetch_lodes(state_abbr):

    base = f"https://lehd.ces.census.gov/data/lodes/LODES8/{state_abbr.lower()}"

    wac_url = f"{base}/wac/{state_abbr.lower()}_wac_S000_JT00_2021.csv.gz"
    rac_url = f"{base}/rac/{state_abbr.lower()}_rac_S000_JT00_2021.csv.gz"

    wac = pd.read_csv(wac_url, compression="gzip")
    rac = pd.read_csv(rac_url, compression="gzip")

    wac["GEOID"] = wac["w_geocode"].astype(str).str[:11]
    rac["GEOID"] = rac["h_geocode"].astype(str).str[:11]

    wac_tract = wac.groupby("GEOID")[["C000","CE01","CE02","CE03"]].sum().reset_index()
    rac_tract = rac.groupby("GEOID")[["C000","CE01","CE02","CE03"]].sum().reset_index()

    wac_tract.columns = ["GEOID","jobs_total","jobs_low","jobs_mid","jobs_high"]
    rac_tract.columns = ["GEOID","workers_total","workers_low","workers_mid","workers_high"]

    df = wac_tract.merge(rac_tract, on="GEOID", how="left")
    df["jobs_inflow_ratio"] = df["jobs_total"] / (df["workers_total"] + 1)

    return df

In [9]:
lodes_ny = fetch_lodes("ny")
print("LODES NY shape:", lodes_ny.shape)
lodes_ny.head()

LODES NY shape: (5364, 10)


,GEOID,jobs_total,jobs_low,jobs_mid,jobs_high,workers_total,workers_low,workers_mid,workers_high,jobs_inflow_ratio
0,36001000100,1207,200,317,690,849.0,166.0,328.0,355.0,1.420000
1,36001000201,672,225,191,256,1101.0,274.0,402.0,425.0,0.609800
2,36001000202,1243,144,346,753,819.0,228.0,303.0,288.0,1.515854
3,36001000301,400,75,119,206,1108.0,279.0,435.0,394.0,0.360685
4,36001000302,12273,1182,2172,8919,1451.0,216.0,402.0,833.0,8.452479


In [18]:
def fetch_smart_location():

    csv_path = os.path.join(RAW_DIR, "SmartLocationDatabaseV3.csv")

    url = "https://edg.epa.gov/data/public/OA/EPA_SmartLocationDatabase_V3_Jan_2021_Final.csv"

    headers = {"User-Agent": "Mozilla/5.0"}

    if not os.path.exists(csv_path):
        print("Downloading Smart Location DB...")
        r = requests.get(url, headers=headers)
        with open(csv_path, "wb") as f:
            f.write(r.content)

    print("Loading Smart Location CSV...")

    df = pd.read_csv(csv_path, encoding="latin1", low_memory=False)

    # 🚀 FIX GEOID FORMAT
    df["GEOID"] = (
        df["GEOID10"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.zfill(11)
    )

    # 🚀 Replace -99999 with NaN
    df.replace(-99999, np.nan, inplace=True)

    # Keep only your 5 states
    state_prefixes = ["06", "08", "50", "11", "36"]
    df = df[df["GEOID"].str[:2].isin(state_prefixes)]

    return df[[
        "GEOID",
        "D3B",
        "D4A",
        "D1C",
        "D2A_JPHH"
    ]]

In [19]:
smart = fetch_smart_location()
print("Smart DB shape:", smart.shape)
smart.head()

Loading Smart Location CSV...
Smart DB shape: (18505, 5)


/var/folders/bq/hl2d1mzx4ln4s9vk722gsfv80000gn/T/ipykernel_6137/3229880765.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["GEOID"] = (


,GEOID,D3B,D4A,D1C,D2A_JPHH
21170,11056871001,0.657314,NaN,0.000562,0.312500
21171,11056870002,3.439966,NaN,0.045960,1.724138
21233,11130302001,132.548411,NaN,3.912396,2.709302
21290,11339655012,1.634005,NaN,0.021925,0.685106
21293,11339655032,5.364258,NaN,0.001657,0.054878


In [20]:
smart["D4A"].describe()

count    11023.000000
mean       319.586622
std        242.113172
min          0.000000
25%        167.640000
50%        257.500000
75%        402.340000
max       1207.000000
Name: D4A, dtype: float64

In [22]:
tracts_ny = tracts_ny.merge(smart, on="GEOID", how="left")

print("After merge:", tracts_ny.shape)
tracts_ny.head()

After merge: (5411, 21)


,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,INTPTLON,geometry,D3B_x,D4A_x,D1C_x,D2A_JPHH_x,D3B_y,D4A_y,D1C_y,D2A_JPHH_y
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,...,-073.9973434,"POLYGON ((-74.00154 40.69279, -74.00132 40.693...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,...,-073.9916018,"POLYGON ((-73.99405 40.6909, -73.99374 40.6915...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,...,-073.9877087,"POLYGON ((-73.99073 40.69305, -73.99045 40.693...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,...,-073.9883586,"POLYGON ((-73.99141 40.69863, -73.99131 40.699...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,...,-074.0159276,"POLYGON ((-74.01867 40.64741, -74.01809 40.647...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Road

In [27]:
tracts_ny.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,INTPTLON,geometry,D3B_x,D4A_x,D1C_x,D2A_JPHH_x,D3B_y,D4A_y,D1C_y,D2A_JPHH_y
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,...,-073.9973434,"POLYGON ((-74.00154 40.69279, -74.00132 40.693...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,...,-073.9916018,"POLYGON ((-73.99405 40.6909, -73.99374 40.6915...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,...,-073.9877087,"POLYGON ((-73.99073 40.69305, -73.99045 40.693...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,...,-073.9883586,"POLYGON ((-73.99141 40.69863, -73.99131 40.699...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,...,-074.0159276,"POLYGON ((-74.01867 40.64741, -74.01809 40.647...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
tracts_ny = tracts_ny.merge(acs_ny, on="GEOID", how="left")
tracts_ny = tracts_ny.merge(lodes_ny, on="GEOID", how="left")
tracts_ny = tracts_ny.merge(smart, on="GEOID", how="left")

tracts_ny.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,jobs_high,workers_total,workers_low,workers_mid,workers_high,jobs_inflow_ratio,D3B,D4A,D1C,D2A_JPHH
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,...,384.0,2041.0,201.0,222.0,1618.0,0.336925,NaN,NaN,NaN,NaN
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,...,6033.0,2520.0,237.0,243.0,2040.0,4.177707,NaN,NaN,NaN,NaN
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,...,55424.0,967.0,73.0,112.0,782.0,64.301653,NaN,NaN,NaN,NaN
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,...,3104.0,1193.0,115.0,169.0,909.0,3.123953,NaN,NaN,NaN,NaN
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,...,872.0,627.0,115.0,253.0,259.0,2.487261,NaN,NaN,NaN,NaN


In [29]:
tracts_ny["ev_adoption_propensity"] = (
    tracts_ny["median_home_value"].rank(pct=True) +
    tracts_ny["jobs_high"].rank(pct=True) +
    tracts_ny["bachelor_plus"].rank(pct=True)
) / 3

tracts_ny["transit_propensity"] = (
    tracts_ny["transit_commuters"].rank(pct=True) +
    tracts_ny["D4A"].rank(pct=True)
) / 2

tracts_ny["home_charging_capacity_index"] = (
    tracts_ny["median_home_value"].rank(pct=True) -
    tracts_ny["transit_commuters"].rank(pct=True)
)

tracts_ny.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,workers_mid,workers_high,jobs_inflow_ratio,D3B,D4A,D1C,D2A_JPHH,ev_adoption_propensity,transit_propensity,home_charging_capacity_index
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,...,222.0,1618.0,0.336925,NaN,NaN,NaN,NaN,0.820469,NaN,0.032431
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,...,243.0,2040.0,4.177707,NaN,NaN,NaN,NaN,0.945503,NaN,-0.072832
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,...,112.0,782.0,64.301653,NaN,NaN,NaN,NaN,0.843001,NaN,0.154281
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,...,169.0,909.0,3.123953,NaN,NaN,NaN,NaN,0.768866,NaN,0.007598
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,...,253.0,259.0,2.487261,NaN,NaN,NaN,NaN,0.600025,NaN,0.311805


In [30]:
print(tracts_ny["GEOID"].dtype)
print(smart["GEOID"].dtype)

str
str


In [31]:
len(set(tracts_ny["GEOID"]).intersection(set(smart["GEOID"])))

0